# DL_04: Hybrid Probability Fusion, Validation Selection & Graph Integration
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook implements **Task 4: Hybrid Probability Fusion and Graph Integration** for the `alpha-sml/causal-nlp-reasoning` pipeline, integrating Phase 1 Applied Machine Learning predictions with Phase 2 Deep Learning predictions into a unified neuro-symbolic causal reasoning pipeline.

### Authoritative BioCause Model-Eligible Dataset Counts
In accordance with source reconstruction audits and strict token-budget limits (<= 384 tokens), the authoritative model-eligible split counts are:

| Split | Candidates | Positive (Causal = 1) | Negative (Non-Causal = 0) | Articles |
| :--- | ---:| ---:| ---:| ---:|
| **Train** | 9,928 | 621 | 9,307 | 12 PMC articles |
| **Validation** | 1,884 | 131 | 1,753 | 4 PMC articles |
| **Test** | 1,394 | 100 | 1,294 | 4 PMC articles |
| **Total Model-Eligible** | 13,206 | 852 | 12,354 | 20 PMC articles |

*(Note: 202 candidates from the 13,408 raw candidate pool were determined ineligible during Task 1 due to overlapping spans or token budget overflow and are tracked in `dl_biocause_reconstruction_diagnostics.csv`).*

---

## 2. Experimental Methodology & Theoretical Background

### Dual-Stream Neuro-Symbolic Architecture
1. **AML Stream (Symbolic / Feature-Engineered)**:
   - Evaluates syntactic dependency paths (SDP), lexical causal trigger cues (active, passive, discourse), positional distances, and TF-IDF n-grams.
   - Primary baseline: `Logistic Regression (Moderate Weight 1:2)` fitted on the training split (`data/processed/aml_biocause_*.csv`), providing strong structural precision.
2. **DL Stream (Neural Representation)**:
   - Sequence-level classification with `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` fine-tuned with special argument markers (`[E1]`, `[/E1]`, `[E2]`, `[/E2]`).
   - Deep contextual representations capture complex multi-clause syntax and implicit causal discourse (`data/processed/dl_biocause_*.csv`).

### Validation Discipline & Probability Fusion
- Continuous prediction probabilities are fused via linear interpolation:
  $$P_{\text{hybrid}} = \alpha \cdot P_{\text{DL}} + (1 - \alpha) \cdot P_{\text{AML}}$$
- **Strict Validation Selection**: The fusion parameter $\alpha \in [0.0, 1.0]$ (step 0.05) and an expanded deterministic decision threshold candidate set ($\tau \in [0.01, 0.99]$ in 0.01 increments combined with empirical validation-probability quantiles in $[0.01, 0.99]$) are swept **strictly on the held-out validation split (1,884 instances)** to maximize validation Positive F1, then Macro-F1, then smaller $\alpha$, then threshold closest to 0.50, then higher $\tau$.
- **Frozen Evaluation**: Once $(\alpha^*, \tau^*)$ are selected, they are permanently frozen and evaluated **exactly once** on the held-out test split (1,394 instances). No test-set data influences hyperparameter selection.

### Symbolic Graph Compilation & Interventional Reasoning
- Surviving positive predictions are formatted into the canonical 17-column `UnifiedRelationRecord` schema (`dl_utils.UNIFIED_RELATION_SCHEMA`).
- Extracted relations are mapped to directed edges $(u \to v)$ in NetworkX.
- Deterministic cycle resolution (`enforce_dag_acyclicity` from `AML/causal_extractors.py`) prunes self-loops, resolves bidirectional conflicts, and breaks feedback cycles using edge confidence scores.
- Acyclic DAGs are compiled into `pgmpy.models.DiscreteBayesianNetwork` to illustrate Pearl's $do$-calculus versus observational conditioning.
- **Critical Guardrail**: DAG acyclicity is a mathematical property required for probabilistic inference, not empirical proof of causal ground truth. Corr2Cause benchmark data is strictly excluded from the BioCause knowledge graph.

In [1]:
# Section 1: Environment Setup, Portable Imports & Self-Contained Task 4 Helpers
import os
import sys
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple, Union

def find_repo_root():
    """Locate repository root across local, Colab, and Kaggle."""
    curr = Path(os.getcwd()).resolve()
    preferred_roots = [
        Path('/content/causal-nlp-reasoning'),
        Path('/workspace/causal-nlp-reasoning'),
        Path('/kaggle/working/causal-nlp-reasoning'),
    ]
    search_roots = [*preferred_roots, curr, Path('/content'), Path('/workspace'), Path('/kaggle/working')]
    checked = set()
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base, *base.parents]
        if base in {Path('/content'), Path('/workspace'), Path('/kaggle/working')}:
            candidates.extend(path.parent.parent for path in base.rglob('DL/dl_utils.py'))
        for p in candidates:
            p = p.resolve()
            if p in checked:
                continue
            checked.add(p)
            if (p / 'AML' / 'causal_extractors.py').is_file() and (p / 'DL' / 'dl_utils.py').is_file():
                return p
    raise FileNotFoundError(
        'Repository root not found. Make the project folder available in the notebook workspace.'
    )

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

import dl_utils
from causal_extractors import enforce_dag_acyclicity, NEGATION_PATTERN
from IPython.display import display
import networkx as nx
import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
)

dl_utils.set_seed(42)
DATA_DIR = ROOT_DIR / 'data' / 'processed'

# --- Self-Contained Task 4 Validation, Fusion, Filtering & Verification Helpers ---
def validate_prediction_dataframe(
    df: pd.DataFrame,
    expected_rows: int,
    split_name: str,
    stream_name: str,
    prob_col: str,
    pred_col: str,
) -> None:
    """Validate prediction dataframe against strict Task 4 input contracts."""
    req_cols = ['candidate_id', 'true_label', prob_col, pred_col, 'split']
    missing_cols = set(req_cols) - set(df.columns)
    if missing_cols:
        raise ValueError(f'[{stream_name} {split_name}] Missing required columns: {sorted(missing_cols)}')
    if len(df) != expected_rows:
        raise ValueError(f'[{stream_name} {split_name}] Row count mismatch: expected {expected_rows}, got {len(df)}')
    if df['candidate_id'].isna().any() or (df['candidate_id'].astype(str).str.strip() == '').any():
        raise ValueError(f'[{stream_name} {split_name}] Found null or empty candidate_id values')
    if df['candidate_id'].duplicated().any():
        raise ValueError(f'[{stream_name} {split_name}] Found duplicate candidate_id values')
    if not (df['split'] == split_name).all():
        raise ValueError(f'[{stream_name} {split_name}] Non-matching split labels in split column')
    if not set(df['true_label'].unique()).issubset({0, 1}):
        raise ValueError(f'[{stream_name} {split_name}] Non-binary true_label values')
    if not set(df[pred_col].unique()).issubset({0, 1}):
        raise ValueError(f'[{stream_name} {split_name}] Non-binary prediction values')
    if not np.isfinite(df[prob_col]).all():
        raise ValueError(f'[{stream_name} {split_name}] Non-finite values in {prob_col}')
    if not df[prob_col].between(0.0, 1.0).all():
        raise ValueError(f'[{stream_name} {split_name}] Probabilities outside [0.0, 1.0] in {prob_col}')

def validate_dual_stream_alignment(df_aml: pd.DataFrame, df_dl: pd.DataFrame, split_name: str) -> None:
    """Assert candidate ID ordering and ground-truth label agreement across AML and DL streams."""
    if len(df_aml) != len(df_dl):
        raise ValueError(f'[{split_name}] Length mismatch across streams: AML ({len(df_aml)}) vs DL ({len(df_dl)})')
    if not (df_aml['candidate_id'].astype(str).values == df_dl['candidate_id'].astype(str).values).all():
        raise ValueError(f'[{split_name}] Candidate ID order or identity mismatch across streams')
    if not (df_aml['true_label'].values == df_dl['true_label'].values).all():
        raise ValueError(f'[{split_name}] Ground truth label discrepancy across streams')

def validate_and_load_authoritative_manifest(
    manifest_path: Union[str, Path],
    split_name: str,
    expected_total: int,
    expected_pos: Optional[int] = None,
    expected_neg: Optional[int] = None,
) -> pd.DataFrame:
    """Validate and load an authoritative Task 1 model-eligible candidate split manifest."""
    path = Path(manifest_path)
    if not path.is_file():
        raise FileNotFoundError(f'Authoritative Task 1 {split_name} manifest not found: {path}')
    df = pd.read_csv(path)
    if 'candidate_id' not in df.columns:
        raise ValueError(f'Manifest {path} missing candidate_id column')
    cids = df['candidate_id'].astype(str).str.strip()
    if df['candidate_id'].isna().any() or (cids == '').any() or cids.duplicated().any():
        raise ValueError(f'Manifest {path} has null, empty, or duplicate candidate_id values')
    if len(df) != expected_total:
        raise ValueError(f'Manifest {path} row count mismatch: expected {expected_total}, got {len(df)}')
    if 'split' in df.columns and not (df['split'].astype(str).str.strip() == split_name).all():
        raise ValueError(f'Manifest {path} contains non-{split_name} rows')
    label_col = 'label' if 'label' in df.columns else ('true_label' if 'true_label' in df.columns else None)
    if label_col is not None:
        if not set(df[label_col].unique()).issubset({0, 1}):
            raise ValueError(f'Manifest {path} contains non-binary labels')
        if expected_pos is not None and int((df[label_col] == 1).sum()) != expected_pos:
            raise ValueError(f'Manifest {path} positive count mismatch')
        if expected_neg is not None and int((df[label_col] == 0).sum()) != expected_neg:
            raise ValueError(f'Manifest {path} negative count mismatch')
    return df

def validate_exact_eligible_membership(
    pred_candidate_ids: Sequence[Any],
    expected_eligible_candidate_ids: Set[str],
    split_name: str,
    stream_name: str,
) -> None:
    """Assert exact set equality between prediction IDs and authoritative model-eligible IDs."""
    pred_set = {str(cid) for cid in pred_candidate_ids}
    if pred_set != expected_eligible_candidate_ids:
        raise ValueError(
            f'[{stream_name} {split_name}] Membership mismatch: '
            f'missing={len(expected_eligible_candidate_ids - pred_set)}, '
            f'extra={len(pred_set - expected_eligible_candidate_ids)}'
        )

def validate_metadata_split_and_label_integrity(
    df_pred: pd.DataFrame,
    df_meta_split: pd.DataFrame,
    split_name: str,
) -> None:
    """Validate prediction candidate labels against isolated split metadata."""
    if 'split' in df_meta_split.columns and not (df_meta_split['split'] == split_name).all():
        raise ValueError(f'Metadata contains non-{split_name} rows')
    meta_indexed = df_meta_split.set_index('candidate_id')
    pred_ids = df_pred['candidate_id'].values
    missing_ids = set(pred_ids) - set(meta_indexed.index)
    if missing_ids:
        raise ValueError(f'[{split_name}] {len(missing_ids)} candidate IDs missing from metadata')
    label_col = 'label' if 'label' in meta_indexed.columns else 'true_label'
    meta_labels = meta_indexed.loc[pred_ids, label_col].values
    if not (df_pred['true_label'].values == meta_labels).all():
        raise ValueError(f'[{split_name}] Candidate labels disagree with manifest metadata')

def build_candidate_thresholds(probs: np.ndarray) -> List[float]:
    """Build deterministic [0.01..0.99] uniform + validation-quantile candidate thresholds."""
    uniform_grid = np.round(np.arange(0.01, 1.00, 0.01), 4)
    quantile_grid = np.round(np.clip(np.quantile(probs, np.linspace(0.01, 0.99, 99)), 0.01, 0.99), 4)
    combined = np.unique(np.concatenate([uniform_grid, quantile_grid]))
    combined = combined[(combined > 0.0) & (combined < 1.0)]
    return [float(t) for t in np.sort(combined)]

def evaluate_threshold_metrics(y_true: np.ndarray, y_prob: np.ndarray, tau: float) -> Dict[str, float]:
    """Evaluate Precision, Recall, Positive F1, and Macro-F1 at threshold tau."""
    pred = y_prob >= float(tau)
    y_pos = y_true == 1
    tp = int(np.sum(pred & y_pos))
    fp = int(np.sum(pred & ~y_pos))
    fn = int(np.sum(~pred & y_pos))
    tn = int(len(y_true) - tp - fp - fn)
    pos_p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    pos_r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pos_f1 = (2.0 * pos_p * pos_r / (pos_p + pos_r)) if (pos_p + pos_r) > 0 else 0.0
    neg_p = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    neg_r = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    neg_f1 = (2.0 * neg_p * neg_r / (neg_p + neg_r)) if (neg_p + neg_r) > 0 else 0.0
    macro_f1 = 0.5 * (neg_f1 + pos_f1)
    return {
        'positive_predictions': int(tp + fp),
        'pos_precision': float(pos_p),
        'pos_recall': float(pos_r),
        'pos_f1': float(pos_f1),
        'macro_f1': float(macro_f1),
    }

def configuration_rank_key(
    f1_pos: float,
    macro_f1: float,
    alpha: float,
    tau: float,
) -> Tuple[float, float, float, float, float]:
    """Deterministic key: Pos F1 -> Macro-F1 -> smaller alpha -> closest to 0.50 -> higher tau."""
    return (
        round(float(f1_pos), 8),
        round(float(macro_f1), 8),
        -round(float(alpha), 6),
        -round(abs(float(tau) - 0.50), 6),
        round(float(tau), 6),
    )

def is_better_configuration(
    cand_f1_pos: float,
    cand_macro_f1: float,
    cand_alpha: float,
    cand_tau: float,
    best_f1_pos: float,
    best_macro_f1: float,
    best_alpha: float,
    best_tau: float,
    rtol: float = 1e-8,
    atol: float = 1e-8,
) -> bool:
    """Compare fusion configurations via Positive F1 -> Macro-F1 -> smaller alpha -> |tau - 0.50| -> higher tau."""
    if best_f1_pos < 0.0:
        return True
    return configuration_rank_key(cand_f1_pos, cand_macro_f1, cand_alpha, cand_tau) > configuration_rank_key(
        best_f1_pos, best_macro_f1, best_alpha, best_tau
    )

def select_validation_fusion_hyperparameters(
    val_y_true: np.ndarray,
    val_p_aml: np.ndarray,
    val_p_dl: np.ndarray,
    alpha_grid: Optional[Sequence[float]] = None,
    tau_grid: Optional[Sequence[float]] = None,
    rtol: float = 1e-8,
    atol: float = 1e-8,
) -> Tuple[float, float, float, float, List[Dict[str, float]]]:
    """Sweep (alpha, tau) strictly on validation data using the deterministic lexicographic rule."""
    if alpha_grid is None:
        alpha_grid = np.round(np.arange(0.0, 1.05, 0.05), 2)
    best_f1_pos, best_macro_f1 = -1.0, -1.0
    best_alpha, best_tau = float(alpha_grid[0]), 0.50
    search_records: List[Dict[str, float]] = []
    for alpha in alpha_grid:
        alpha_val = float(alpha)
        p_hyb = dl_utils.fuse_probabilities(prob_aml=val_p_aml, prob_dl=val_p_dl, alpha=alpha_val)
        cand_taus = build_candidate_thresholds(p_hyb) if tau_grid is None else [float(t) for t in tau_grid]
        for tau in cand_taus:
            tau_val = float(tau)
            m = evaluate_threshold_metrics(val_y_true, p_hyb, tau_val)
            search_records.append({
                'alpha': alpha_val,
                'tau': tau_val,
                'positive_predictions': m['positive_predictions'],
                'pos_precision': m['pos_precision'],
                'pos_recall': m['pos_recall'],
                'pos_f1': m['pos_f1'],
                'macro_f1': m['macro_f1'],
            })
            if is_better_configuration(
                m['pos_f1'], m['macro_f1'], alpha_val, tau_val,
                best_f1_pos, best_macro_f1, best_alpha, best_tau, rtol=rtol, atol=atol,
            ):
                best_f1_pos, best_macro_f1 = m['pos_f1'], m['macro_f1']
                best_alpha, best_tau = alpha_val, tau_val
    p_hyb_best = dl_utils.fuse_probabilities(prob_aml=val_p_aml, prob_dl=val_p_dl, alpha=best_alpha)
    y_pred_best = (p_hyb_best >= best_tau).astype(int)
    _, _, sk_f1_pos, _ = precision_recall_fscore_support(val_y_true, y_pred_best, average='binary', zero_division=0)
    _, _, sk_macro_f1, _ = precision_recall_fscore_support(val_y_true, y_pred_best, average='macro', zero_division=0)
    assert np.isclose(best_f1_pos, float(sk_f1_pos), atol=1e-8)
    assert np.isclose(best_macro_f1, float(sk_macro_f1), atol=1e-8)
    return best_alpha, best_tau, best_f1_pos, best_macro_f1, search_records

def apply_linguistic_filters(
    df: pd.DataFrame,
    max_between_chars: int = 400,
    pred_col: str = 'raw_pred',
) -> pd.DataFrame:
    """Apply Phase 1 negation pattern and 400-char distance filter with explicit metadata validation."""
    if 'between_text' not in df.columns:
        raise ValueError("Missing 'between_text' column required for linguistic sanity checks")
    df_out = df.copy()
    neg_flags, dist_flags, reasons = [], [], []
    for idx, r in df_out.iterrows():
        raw_val = r['between_text']
        if raw_val is None or pd.isna(raw_val):
            raise ValueError(f"Missing between_text for candidate {r.get('candidate_id', idx)}")
        b_txt = str(raw_val)
        if b_txt.strip().lower() == 'nan':
            raise ValueError(f"Invalid between_text ('nan') for candidate {r.get('candidate_id', idx)}")
        if b_txt == '' and int(r.get('char_distance', -1)) != 0:
            raise ValueError(f"Empty between_text with non-zero distance for candidate {r.get('candidate_id', idx)}")
        is_neg = bool(NEGATION_PATTERN.search(b_txt))
        is_dist = len(b_txt) > max_between_chars
        neg_flags.append(is_neg)
        dist_flags.append(is_dist)
        r_list = []
        if is_neg:
            r_list.append('negation_detected')
        if is_dist:
            r_list.append('distance_window_exceeded')
        reasons.append(';'.join(r_list) if r_list else 'none')
    df_out['flag_negation'] = neg_flags
    df_out['flag_distance'] = dist_flags
    df_out['pruned_reason'] = reasons
    df_out['sanitized_pred'] = ((df_out[pred_col] == 1) & (~df_out['flag_negation']) & (~df_out['flag_distance'])).astype(int)
    return df_out

def compute_linguistic_diagnostics(
    df: pd.DataFrame,
    pred_col: str = 'raw_hybrid_pred',
    sanitized_col: str = 'sanitized_pred',
) -> Dict[str, int]:
    """Compute non-overlapping rejection breakdown across negation and distance filters."""
    is_pos = df[pred_col] == 1
    n_initial = int(is_pos.sum())
    n_neg_only = int((is_pos & df['flag_negation'] & (~df['flag_distance'])).sum())
    n_dist_only = int((is_pos & (~df['flag_negation']) & df['flag_distance']).sum())
    n_both = int((is_pos & df['flag_negation'] & df['flag_distance']).sum())
    n_total_rejected = n_neg_only + n_dist_only + n_both
    n_surviving = int((df[sanitized_col] == 1).sum())
    if n_initial != n_surviving + n_total_rejected:
        raise ValueError(f'Linguistic diagnostic mismatch: {n_initial} != {n_surviving} + {n_total_rejected}')
    return {
        'initial_positive': n_initial,
        'rejected_negation_only': n_neg_only,
        'rejected_distance_only': n_dist_only,
        'rejected_both': n_both,
        'total_unique_rejected': n_total_rejected,
        'surviving_positive': n_surviving,
    }

def verify_prediction_export(
    filepath: Union[str, Path],
    expected_rows: int,
    split_name: str,
    source_candidate_ids: Sequence[Any],
    source_true_labels: Sequence[int],
    source_aml_probs: Sequence[float],
    source_dl_probs: Sequence[float],
    frozen_alpha: float,
    frozen_tau: float,
    source_aml_preds: Optional[Sequence[int]] = None,
    source_dl_preds: Optional[Sequence[int]] = None,
    rtol: float = 1e-5,
    atol: float = 1e-5,
    include_sanitized_col: bool = False,
) -> pd.DataFrame:
    """Read back and verify exported hybrid prediction CSV against source arrays and frozen config."""
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f'Exported prediction file not found: {path}')
    df_exp = pd.read_csv(path)
    if len(df_exp) != expected_rows:
        raise ValueError(f'[{split_name} Export] Row count mismatch: expected {expected_rows}, got {len(df_exp)}')
    req_cols = ['candidate_id', 'true_label', 'aml_prob', 'dl_prob', 'hybrid_prob', 'aml_pred', 'dl_pred', 'hybrid_pred', 'split']
    if include_sanitized_col:
        req_cols.append('sanitized_pred')
    missing_cols = set(req_cols) - set(df_exp.columns)
    if missing_cols:
        raise ValueError(f'[{split_name} Export] Missing columns: {sorted(missing_cols)}')
    for col in req_cols:
        if df_exp[col].isna().any():
            raise ValueError(f'[{split_name} Export] Found nulls in {col}')
    if not (df_exp['candidate_id'].astype(str).values == np.array(source_candidate_ids).astype(str)).all():
        raise ValueError(f'[{split_name} Export] Candidate ID ordering mismatch')
    if not (df_exp['true_label'].values == np.array(source_true_labels)).all():
        raise ValueError(f'[{split_name} Export] true_label mismatch')
    for p_col in ['aml_prob', 'dl_prob', 'hybrid_prob']:
        if not np.isfinite(df_exp[p_col]).all() or not df_exp[p_col].between(0.0, 1.0).all():
            raise ValueError(f'[{split_name} Export] Invalid probability values in {p_col}')
    src_aml_p = np.asarray(source_aml_probs, dtype=float)
    src_dl_p = np.asarray(source_dl_probs, dtype=float)
    if not np.allclose(df_exp['aml_prob'].values, src_aml_p, rtol=rtol, atol=atol):
        raise ValueError(f'[{split_name} Export] Exported aml_prob mismatch')
    if not np.allclose(df_exp['dl_prob'].values, src_dl_p, rtol=rtol, atol=atol):
        raise ValueError(f'[{split_name} Export] Exported dl_prob mismatch')
    recomputed = frozen_alpha * src_dl_p + (1.0 - frozen_alpha) * src_aml_p
    if not np.allclose(df_exp['hybrid_prob'].values, recomputed, rtol=rtol, atol=atol):
        raise ValueError(f'[{split_name} Export] Exported hybrid_prob does not match fusion formula')
    for pred_col in ['aml_pred', 'dl_pred', 'hybrid_pred']:
        if not set(df_exp[pred_col].unique()).issubset({0, 1}):
            raise ValueError(f'[{split_name} Export] Non-binary values in {pred_col}')
    if source_aml_preds is not None and not (df_exp['aml_pred'].values == np.asarray(source_aml_preds, dtype=int)).all():
        raise ValueError(f'[{split_name} Export] Exported aml_pred mismatch')
    if source_dl_preds is not None and not (df_exp['dl_pred'].values == np.asarray(source_dl_preds, dtype=int)).all():
        raise ValueError(f'[{split_name} Export] Exported dl_pred mismatch')
    if not (df_exp['hybrid_pred'].values == (df_exp['hybrid_prob'].values >= frozen_tau).astype(int)).all():
        raise ValueError(f'[{split_name} Export] hybrid_pred does not match threshold')
    if include_sanitized_col and not set(df_exp['sanitized_pred'].unique()).issubset({0, 1}):
        raise ValueError(f'[{split_name} Export] Non-binary sanitized_pred values')
    if not (df_exp['split'] == split_name).all():
        raise ValueError(f'[{split_name} Export] Invalid split values')
    return df_exp

def verify_relation_export(
    filepath: Union[str, Path],
    expected_count: int,
    sanitized_candidate_ids: Sequence[Any],
    traceability_map: Optional[Dict[str, str]] = None,
) -> pd.DataFrame:
    """Verify 17-column UnifiedRelationRecord export with SHA-256 candidate traceability."""
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f'Exported relation file not found: {path}')
    df_rels = pd.read_csv(path)
    if list(df_rels.columns) != dl_utils.UNIFIED_RELATION_SCHEMA:
        raise ValueError(f'Relation schema mismatch: {list(df_rels.columns)}')
    if len(df_rels) != expected_count or len(df_rels) != len(sanitized_candidate_ids):
        raise ValueError('Relation export count mismatch with sanitized positives')
    for f in ['relation_id', 'article_id', 'section_id', 'cause_id', 'effect_id']:
        if df_rels[f].isna().any() or (df_rels[f].astype(str).str.strip() == '').any():
            raise ValueError(f'Null or empty values in {f}')
    if df_rels['relation_id'].duplicated().any():
        raise ValueError('Duplicate relation_id values found')
    if (df_rels['cause_id'] == df_rels['effect_id']).any():
        raise ValueError('Self-referential relation (cause_id == effect_id) found')
    if not (df_rels['direction'] == 'cause->effect').all() or not (df_rels['split'] == 'test').all():
        raise ValueError('Invalid direction or split in relation export')
    conf = pd.to_numeric(df_rels['confidence'], errors='coerce')
    if conf.isna().any() or not np.isfinite(conf).all() or not conf.between(0.0, 1.0).all():
        raise ValueError('Invalid confidence values in relation export')
    if (df_rels['cause_start'] >= df_rels['cause_end']).any() or (df_rels['effect_start'] >= df_rels['effect_end']).any():
        raise ValueError('Invalid start/end span offsets in relation export')
    expected_rel_ids = {dl_utils.generate_relation_id(str(cid)) for cid in sanitized_candidate_ids}
    actual_rel_ids = set(df_rels['relation_id'].astype(str))
    if actual_rel_ids != expected_rel_ids or (traceability_map is not None and actual_rel_ids != set(traceability_map.values())):
        raise ValueError('Relation ID traceability mismatch')
    return df_rels

def compile_discrete_bayesian_network(G_dag: nx.DiGraph) -> DiscreteBayesianNetwork:
    """Compile G_dag into a pgmpy DiscreteBayesianNetwork preserving all isolated and connected nodes."""
    bn = DiscreteBayesianNetwork()
    bn.add_nodes_from(G_dag.nodes())
    bn.add_edges_from(G_dag.edges())
    assert set(bn.nodes()) == set(G_dag.nodes()), 'Node set mismatch in Bayesian network compilation'
    assert set(bn.edges()) == set(G_dag.edges()), 'Edge set mismatch in Bayesian network compilation'
    assert nx.is_directed_acyclic_graph(bn), 'Compiled Bayesian network is not acyclic'
    return bn

def verify_dag_and_pruning_diagnostics(
    G_pred: nx.DiGraph,
    G_dag: nx.DiGraph,
    audit_log: List[Dict[str, Any]],
    diag_filepath: Union[str, Path],
    dag_edges_filepath: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """Verify graph acyclicity and reconcile persisted pruning diagnostics JSON."""
    assert nx.is_directed_acyclic_graph(G_dag), 'Final graph G_dag is not a DAG'
    n_initial_edges = G_pred.number_of_edges()
    n_final_edges = G_dag.number_of_edges()
    n_pruned = len(audit_log)
    assert n_initial_edges == n_final_edges + n_pruned
    path = Path(diag_filepath)
    if not path.is_file():
        raise FileNotFoundError(f'Diagnostics file not found: {path}')
    with open(path) as f:
        diag = json.load(f)
    assert diag['initial_candidate_edges'] == n_initial_edges
    assert diag['final_dag_edges'] == n_final_edges
    assert diag['pruned_edges_count'] == n_pruned
    n_self = sum(1 for e in audit_log if e.get('reason') == 'self_loop')
    n_bidir = sum(1 for e in audit_log if 'bidirectional_conflict' in e.get('reason', ''))
    n_cycle = sum(1 for e in audit_log if 'cycle' in e.get('reason', ''))
    pb = diag.get('pruning_breakdown', {})
    assert pb.get('self_loops', 0) == n_self
    assert pb.get('bidirectional_conflicts', 0) == n_bidir
    assert pb.get('directed_cycle_feedback', 0) == n_cycle
    for stage in ['before_pruning', 'after_pruning']:
        metrics = diag['edge_evaluation'][stage]
        for m in ['Precision', 'Recall', 'F1']:
            assert np.isfinite(metrics[m]) and 0.0 <= metrics[m] <= 1.0
    if dag_edges_filepath is not None:
        df_edges = pd.read_csv(dag_edges_filepath)
        edge_pairs = set(zip(df_edges['source'].astype(str), df_edges['target'].astype(str)))
        expected_pairs = {(str(u), str(v)) for u, v in G_dag.edges()}
        assert edge_pairs == expected_pairs, 'DAG edges CSV does not match NetworkX G_dag'
    return diag

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## 3. Expected Inputs & Prediction Contracts

| Stream / Manifest | File Path | Expected Rows | Required Columns / Description |
| :--- | :--- | :---: | :--- |
| **AML Validation Preds** | `data/processed/aml_biocause_validation_predictions.csv` | 1,884 | `candidate_id, true_label, aml_prob, aml_pred, split` |
| **DL Validation Preds** | `data/processed/dl_biocause_validation_predictions.csv` | 1,884 | `candidate_id, true_label, dl_prob, dl_pred, split` |
| **Task 1 Val Split Manifest** | `data/processed/dl_biocause_val.csv` | 1,884 | Authoritative model-eligible validation candidate manifest |
| **AML Test Preds** | `data/processed/aml_biocause_test_predictions.csv` | 1,394 | `candidate_id, true_label, aml_prob, aml_pred, split` |
| **DL Test Preds** | `data/processed/dl_biocause_test_predictions.csv` | 1,394 | `candidate_id, true_label, dl_prob, dl_pred, split` |
| **Task 1 Test Split Manifest**| `data/processed/dl_biocause_test.csv` | 1,394 | Authoritative model-eligible test candidate manifest |
| **Candidate Metadata** | `data/processed/biocause_candidates.csv` | 13,408 | 17 metadata columns including `between_text` |
| **Gold Relations** | `data/processed/biocause_relations.csv` | 852 | Gold directed relation annotations (100 test) |

### Strict Execution Sequence & Evaluation Discipline
To guarantee zero data snooping or subtle hyperparameter leakage:
1. **Section 2**: Ingest and validate **validation predictions and isolated validation candidate metadata only**. Test metadata, test labels, test predictions, and gold relations are strictly NOT loaded or inspected during validation selection.
2. **Section 3**: Search and freeze ($\alpha^*, \tau^*$) strictly on validation data using the documented lexicographic rule (positive F1, then Macro-F1, then smaller $\alpha$, then threshold closest to 0.50, then higher $\tau$).
3. **Section 4**: Ingest test predictions, isolated test candidate metadata, and gold test relations **only after** hyperparameter freezing, and evaluate exactly once.
4. **Pure Baselines**: Pure AML and Pure DL baselines are evaluated using thresholds selected under the same validation objective and tie-breaking procedure as the hybrid threshold (alongside fixed reference thresholds).

### Population Reference: Raw Candidates vs Model-Eligible Candidates
- **Raw Candidate Pool**: 13,408 total (Train: 10,086, Validation: 1,914, Test: 1,408).
- **Model-Eligible Population**: 13,206 total (Train: 9,928, Validation: 1,884, Test: 1,394).
- *202 candidates (Train: 158, Val: 30, Test: 14) were disqualified during Task 1 token-budget audits and are tracked in diagnostics.*

In [2]:
# Section 2: Validation Data Ingestion & Dual-Stream Alignment

def resolve_input_path(filename: str) -> Path:
    candidates = [DATA_DIR / filename, ROOT_DIR / 'data' / 'processed' / filename]
    if Path('/content').exists():
        candidates.extend(Path('/content').rglob(filename))
    for c in candidates:
        if c.exists():
            return c
    return DATA_DIR / filename

# 1. Verify existence of ALL required files upfront (without parsing test contents before freeze)
aml_val_path = resolve_input_path('aml_biocause_validation_predictions.csv')
dl_val_path = resolve_input_path('dl_biocause_validation_predictions.csv')
aml_test_path = resolve_input_path('aml_biocause_test_predictions.csv')
dl_test_path = resolve_input_path('dl_biocause_test_predictions.csv')
val_manifest_path = resolve_input_path('dl_biocause_val.csv')
test_manifest_path = resolve_input_path('dl_biocause_test.csv')
cands_meta_path = resolve_input_path('biocause_candidates.csv')
relations_path = resolve_input_path('biocause_relations.csv')

required_files = {
    'AML Validation Predictions': aml_val_path,
    'DL Validation Predictions': dl_val_path,
    'AML Test Predictions': aml_test_path,
    'DL Test Predictions': dl_test_path,
    'Prepared Val Split': val_manifest_path,
    'Prepared Test Split': test_manifest_path,
    'BioCause Candidates Metadata': cands_meta_path,
    'BioCause Gold Relations': relations_path,
}
missing_files = [name for name, p in required_files.items() if not p.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required DL_04 input files: {missing_files}')

# 2. Ingest VALIDATION predictions and isolated validation manifest ONLY
df_aml_val = pd.read_csv(aml_val_path)
df_dl_val = pd.read_csv(dl_val_path)

val_counts = dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['validation']
df_val_manifest = validate_and_load_authoritative_manifest(
    val_manifest_path,
    'validation',
    val_counts['total'],
    expected_pos=val_counts['positive'],
    expected_neg=val_counts['negative'],
)
expected_val_ids = set(df_val_manifest['candidate_id'].astype(str))

validate_prediction_dataframe(df_aml_val, val_counts['total'], 'validation', 'AML', 'aml_prob', 'aml_pred')
validate_prediction_dataframe(df_dl_val, val_counts['total'], 'validation', 'DL', 'dl_prob', 'dl_pred')
validate_dual_stream_alignment(df_aml_val, df_dl_val, 'Validation')
validate_exact_eligible_membership(df_aml_val['candidate_id'], expected_val_ids, 'validation', 'AML')
validate_exact_eligible_membership(df_dl_val['candidate_id'], expected_val_ids, 'validation', 'DL')
validate_metadata_split_and_label_integrity(df_aml_val, df_val_manifest, 'validation')
validate_metadata_split_and_label_integrity(df_dl_val, df_val_manifest, 'validation')

# 3. Assert strict isolation: No test metadata, predictions, or gold annotations loaded
assert (df_val_manifest['split'] == 'validation').all(), 'Leaked non-validation metadata'
assert 'df_cands_test_meta' not in globals()
assert 'df_aml_test' not in globals()
assert 'df_dl_test' not in globals()
assert 'df_gold_relations' not in globals()

val_stream_summary_df = pd.DataFrame([
    {
        'Stream': 'AML Validation',
        'Candidates': int(len(df_aml_val)),
        'Positive_Labels': int((df_aml_val['true_label'] == 1).sum()),
        'Prob_Min': round(float(df_aml_val['aml_prob'].min()), 4),
        'Prob_Median': round(float(df_aml_val['aml_prob'].median()), 4),
        'Prob_Mean': round(float(df_aml_val['aml_prob'].mean()), 4),
        'Prob_Max': round(float(df_aml_val['aml_prob'].max()), 4),
    },
    {
        'Stream': 'PubMedBERT Validation',
        'Candidates': int(len(df_dl_val)),
        'Positive_Labels': int((df_dl_val['true_label'] == 1).sum()),
        'Prob_Min': round(float(df_dl_val['dl_prob'].min()), 4),
        'Prob_Median': round(float(df_dl_val['dl_prob'].median()), 4),
        'Prob_Mean': round(float(df_dl_val['dl_prob'].mean()), 4),
        'Prob_Max': round(float(df_dl_val['dl_prob'].max()), 4),
    },
])
val_stream_summary_df

,Stream,Candidates,Positive_Labels,Prob_Min,Prob_Median,Prob_Mean,Prob_Max
0,AML Validation,1884,131,0.0001,0.0310,0.1192,0.9340
1,PubMedBERT Validation,1884,131,0.0013,0.0059,0.0335,0.3182


## 4. Validation-Selected Probability-Score Fusion

### Linear Fusion Formula
For every candidate $i$, continuous probability estimates are combined via:
$$P_{\text{hybrid}}^{(i)} = \alpha \cdot P_{\text{DL}}^{(i)} + (1 - \alpha) \cdot P_{\text{AML}}^{(i)}$$

### Hyperparameter Search Protocol
- **Evaluation Discipline**: Model parameters $(\alpha, \tau)$ are tuned strictly on the **validation split (1,884 candidates)**. No test data is touched during tuning.
- **$\alpha$ Sweep**: Grid $\alpha \in [0.0, 1.0]$ in increments of 0.05 (21 configurations), spanning pure AML ($\alpha=0.0$), balanced ensembles ($\alpha=0.50$), and pure DL ($\alpha=1.0$).
- **Threshold $\tau$ Sweep**: Expanded deterministic candidate set combining $\tau \in [0.01, 0.99]$ in increments of 0.01 with empirical validation-probability quantiles clipped to $[0.01, 0.99]$.
- **Selection Objective & Tie-Breaking**: Maximize Validation Positive Causal F1 ($F_{1, \text{pos}}$), then Validation Macro-F1, then smaller $\alpha$, then threshold closest to 0.50, then higher $\tau$.
- **Freezing**: Once optimal $(\alpha^*, \tau^*)$ are identified, they are frozen for test evaluation.

In [3]:
# Section 3: Validation-Selected Probability-Score Fusion
alpha_grid = np.round(np.arange(0.0, 1.05, 0.05), 2)

val_y_true = df_aml_val['true_label'].to_numpy()
val_p_aml = df_aml_val['aml_prob'].to_numpy()
val_p_dl = df_dl_val['dl_prob'].to_numpy()

best_alpha, best_tau, best_val_f1_pos, best_val_macro_f1, search_records = select_validation_fusion_hyperparameters(
    val_y_true, val_p_aml, val_p_dl, alpha_grid=alpha_grid, tau_grid=None
)
FROZEN_ALPHA = float(best_alpha)
FROZEN_TAU = float(best_tau)

# Select AML-only and DL-only thresholds using the identical validation objective and tie-breaker
_, AML_VALIDATION_TAU, aml_val_pos_f1, aml_val_macro_f1, _ = select_validation_fusion_hyperparameters(
    val_y_true, val_p_aml, val_p_dl, alpha_grid=[0.0], tau_grid=None
)
_, DL_VALIDATION_TAU, dl_val_pos_f1, dl_val_macro_f1, _ = select_validation_fusion_hyperparameters(
    val_y_true, val_p_aml, val_p_dl, alpha_grid=[1.0], tau_grid=None
)
AML_VALIDATION_TAU = float(AML_VALIDATION_TAU)
DL_VALIDATION_TAU = float(DL_VALIDATION_TAU)

assert np.array_equal((val_p_dl >= DL_VALIDATION_TAU).astype(int), df_dl_val['dl_pred'].to_numpy()), (
    'DL validation threshold diverges from DL_03 validation dl_pred'
)
dl_metrics_path = DATA_DIR / 'dl_biocause_metrics.json'
if dl_metrics_path.is_file():
    dl_metrics_meta = json.loads(dl_metrics_path.read_text())
    assert np.isclose(DL_VALIDATION_TAU, float(dl_metrics_meta['selected_threshold']), atol=1e-4), (
        f"DL_04 validation threshold {DL_VALIDATION_TAU} != DL_03 selected_threshold {dl_metrics_meta['selected_threshold']}"
    )

val_p_hybrid_best = dl_utils.fuse_probabilities(val_p_aml, val_p_dl, FROZEN_ALPHA)

def summarize_validation_config(config_name: str, alpha_val: float, tau_val: float) -> Dict[str, Any]:
    p_cfg = dl_utils.fuse_probabilities(val_p_aml, val_p_dl, float(alpha_val))
    m = evaluate_threshold_metrics(val_y_true, p_cfg, float(tau_val))
    return {
        'Configuration': config_name,
        'Alpha': round(float(alpha_val), 2),
        'Selected_Threshold': round(float(tau_val), 4),
        'Positive_Predictions': int(m['positive_predictions']),
        'Val_Pos_Precision': round(float(m['pos_precision']), 4),
        'Val_Pos_Recall': round(float(m['pos_recall']), 4),
        'Val_Pos_F1': round(float(m['pos_f1']), 4),
        'Val_Macro_F1': round(float(m['macro_f1']), 4),
        'Delta_Pos_F1_vs_AML': round(float(m['pos_f1'] - aml_val_pos_f1), 4),
    }

val_selection_summary_df = pd.DataFrame([
    summarize_validation_config('AML-only (fixed 0.50)', 0.0, 0.50),
    summarize_validation_config('AML-only (validation-selected)', 0.0, AML_VALIDATION_TAU),
    summarize_validation_config('PubMedBERT-only (validation-selected)', 1.0, DL_VALIDATION_TAU),
    summarize_validation_config('PubMedBERT-only (fixed 0.50)', 1.0, 0.50),
    summarize_validation_config('Selected Hybrid (frozen)', FROZEN_ALPHA, FROZEN_TAU),
])
val_selection_summary_df

,Configuration,Alpha,Selected_Threshold,Positive_Predictions,Val_Pos_Precision,Val_Pos_Recall,Val_Pos_F1,Val_Macro_F1,Delta_Pos_F1_vs_AML
0,AML-only (fixed 0.50),0.00,0.5000,129,0.5969,0.5878,0.5923,0.7810,-0.0053
1,AML-only (validation-selected),0.00,0.5200,120,0.6250,0.5725,0.5976,0.7844,0.0000
2,PubMedBERT-only (validation-selected),1.00,0.0273,377,0.2493,0.7176,0.3701,0.6360,-0.2275
3,PubMedBERT-only (fixed 0.50),1.00,0.5000,0,0.0000,0.0000,0.0000,0.4820,-0.5976
4,Selected Hybrid (frozen),0.45,0.3200,119,0.6723,0.6107,0.6400,0.8072,0.0424


## 5. Held-Out Test Evaluation (Single Evaluation Run)

With $(\alpha^*, \tau^*)$ frozen strictly from validation data, we evaluate once on the held-out test split (1,394 candidates, 100 gold positive causal relations).

We contrast:
1. **Pure AML Baseline**: Logistic Regression (Moderate Weight 1:2)
2. **Pure PubMedBERT Baseline**: Task 3 fine-tuned model
3. **Selected Hybrid Model**: Linear probability fusion with frozen parameters

In [4]:
# Section 4: Held-Out Test Data Ingestion & Frozen Evaluation
assert np.isfinite(FROZEN_ALPHA) and 0.0 <= FROZEN_ALPHA <= 1.0
assert np.isfinite(FROZEN_TAU) and 0.0 < FROZEN_TAU < 1.0
assert np.isfinite(AML_VALIDATION_TAU) and 0.0 < AML_VALIDATION_TAU < 1.0
assert np.isfinite(DL_VALIDATION_TAU) and 0.0 < DL_VALIDATION_TAU < 1.0

df_aml_test = pd.read_csv(aml_test_path)
df_dl_test = pd.read_csv(dl_test_path)
df_gold_relations = pd.read_csv(relations_path, keep_default_na=False)
df_cands_test_meta = pd.read_csv(cands_meta_path, keep_default_na=False).query("split == 'test'").copy()

test_counts = dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']
df_test_manifest = validate_and_load_authoritative_manifest(
    test_manifest_path,
    'test',
    test_counts['total'],
    expected_pos=test_counts['positive'],
    expected_neg=test_counts['negative'],
)
expected_test_ids = set(df_test_manifest['candidate_id'].astype(str))
for frame, stream, prob, pred in [
    (df_aml_test, 'AML', 'aml_prob', 'aml_pred'),
    (df_dl_test, 'DL', 'dl_prob', 'dl_pred'),
]:
    validate_prediction_dataframe(frame, test_counts['total'], 'test', stream, prob, pred)
    validate_exact_eligible_membership(frame['candidate_id'], expected_test_ids, 'test', stream)

validate_dual_stream_alignment(df_aml_test, df_dl_test, 'Test')
validate_metadata_split_and_label_integrity(df_aml_test, df_test_manifest, 'test')
validate_metadata_split_and_label_integrity(df_dl_test, df_test_manifest, 'test')
assert np.array_equal(
    (df_dl_test['dl_prob'].to_numpy() >= DL_VALIDATION_TAU).astype(int),
    df_dl_test['dl_pred'].to_numpy(),
), 'DL test predictions diverge from validation-selected DL threshold'

test_y_true = df_aml_test['true_label'].to_numpy()
test_p_aml = df_aml_test['aml_prob'].to_numpy()
test_p_dl = df_dl_test['dl_prob'].to_numpy()
test_p_hybrid = dl_utils.fuse_probabilities(test_p_aml, test_p_dl, FROZEN_ALPHA)
test_y_pred_hybrid = (test_p_hybrid >= FROZEN_TAU).astype(int)
assert len(test_p_hybrid) == len(expected_test_ids)

def evaluate_stream(y_true, probabilities, threshold, name):
    predictions = (probabilities >= threshold).astype(int)
    cm_arr = confusion_matrix(y_true, predictions, labels=[0, 1])
    tn, fp, fn, tp = [int(x) for x in cm_arr.ravel()]
    p, r, f1, _ = precision_recall_fscore_support(y_true, predictions, average='binary', zero_division=0)
    _, _, macro_f1, _ = precision_recall_fscore_support(y_true, predictions, average='macro', zero_division=0)
    return {
        'Stream': name,
        'Threshold': float(threshold),
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Accuracy': float(accuracy_score(y_true, predictions)),
        'Pos Precision': float(p),
        'Pos Recall': float(r),
        'Pos F1': float(f1),
        'Macro F1': float(macro_f1),
        'ROC-AUC': float(roc_auc_score(y_true, probabilities)),
        'PR-AUC': float(average_precision_score(y_true, probabilities)),
        'Confusion Matrix': cm_arr,
    }

eval_aml = evaluate_stream(test_y_true, test_p_aml, 0.50, 'Pure AML (fixed 0.50)')
eval_aml_validation = evaluate_stream(test_y_true, test_p_aml, AML_VALIDATION_TAU, 'AML-only (validation-selected)')
eval_dl_validation = evaluate_stream(test_y_true, test_p_dl, DL_VALIDATION_TAU, 'Pure PubMedBERT (validation-selected)')
eval_dl_default = evaluate_stream(test_y_true, test_p_dl, 0.50, 'Pure PubMedBERT (fixed 0.50)')
eval_hybrid_selected = evaluate_stream(
    test_y_true, test_p_hybrid, FROZEN_TAU, f'Hybrid Model (alpha={FROZEN_ALPHA:.2f})'
)
test_summary = [eval_aml, eval_aml_validation, eval_dl_validation, eval_dl_default, eval_hybrid_selected]
test_summary_cols = [
    'Stream', 'Threshold', 'TP', 'FP', 'FN', 'TN', 'Accuracy',
    'Pos Precision', 'Pos Recall', 'Pos F1', 'Macro F1', 'ROC-AUC', 'PR-AUC',
    'Delta Pos F1 vs AML',
]
test_summary_rows = []
for row in test_summary:
    entry = {k: row[k] for k in test_summary_cols if k != 'Delta Pos F1 vs AML'}
    entry['Delta Pos F1 vs AML'] = float(row['Pos F1'] - eval_aml_validation['Pos F1'])
    test_summary_rows.append(entry)
test_summary_df = pd.DataFrame(test_summary_rows)
test_summary_df['Threshold'] = test_summary_df['Threshold'].round(4)
for col in ['Accuracy', 'Pos Precision', 'Pos Recall', 'Pos F1', 'Macro F1', 'ROC-AUC', 'PR-AUC', 'Delta Pos F1 vs AML']:
    test_summary_df[col] = test_summary_df[col].round(4)
test_summary_df

,Stream,Threshold,TP,FP,FN,TN,Accuracy,Pos Precision,Pos Recall,Pos F1,Macro F1,ROC-AUC,PR-AUC,Delta Pos F1 vs AML
0,Pure AML (fixed 0.50),0.5000,52,33,48,1261,0.9419,0.6118,0.52,0.5622,0.7655,0.9431,0.6119,0.0053
1,AML-only (validation-selected),0.5200,49,27,51,1267,0.9440,0.6447,0.49,0.5568,0.7635,0.9431,0.6119,0.0000
2,Pure PubMedBERT (validation-selected),0.0273,80,228,20,1066,0.8221,0.2597,0.80,0.3922,0.6440,0.8809,0.3130,-0.1647
3,Pure PubMedBERT (fixed 0.50),0.5000,0,0,100,1294,0.9283,0.0000,0.00,0.0000,0.4814,0.8809,0.3130,-0.5568
4,Hybrid Model (alpha=0.45),0.3200,57,36,43,1258,0.9433,0.6129,0.57,0.5907,0.7801,0.9448,0.6364,0.0339


## 6. Linguistic Sanity Validation & Candidate Pruning

Even with strong neural-statistical confidence, candidate predictions can violate domain-specific linguistic invariants:
1. **Explicit Negation Filter**:
   - **Rule**: If the intervening text between cause and effect contains explicit negation cues matching `NEGATION_PATTERN` (e.g., `not`, `failed to`, `unable to`, `neither/nor`), the candidate relation is negated or inverted.
   - **Rationale**: Biological causal extraction seeks affirmative directed regulatory relationships. Negated assertions must not enter the knowledge graph as positive regulatory edges.
2. **Argument Distance Window Constraint**:
   - **Rule**: If the intervening text length exceeds 400 characters (`len(between_text) > 400`), the candidate is pruned.
   - **Rationale**: Long-range syntactic dependencies across multi-sentence or distal contexts suffer from parse degradation and false-positive entity pairing.

In [5]:
# Section 5: Linguistic Sanity Validation & Candidate Pruning
df_test_full = df_aml_test[['candidate_id', 'true_label']].copy()
df_test_full['aml_prob'] = test_p_aml
df_test_full['dl_prob'] = test_p_dl
df_test_full['hybrid_prob'] = test_p_hybrid
df_test_full['raw_hybrid_pred'] = test_y_pred_hybrid
df_test_full['split'] = 'test'

meta_cols = [
    'article_id', 'section_id', 'arg1_id', 'arg1_text', 'arg1_start', 'arg1_end',
    'arg2_id', 'arg2_text', 'arg2_start', 'arg2_end', 'char_distance', 'arg1_before_arg2', 'between_text'
]
df_test_full = df_test_full.merge(df_cands_test_meta[['candidate_id'] + meta_cols], on='candidate_id', how='inner')
assert len(df_test_full) == len(df_aml_test), f'Metadata merge row count mismatch: {len(df_test_full)} vs {len(df_aml_test)}'

MAX_BETWEEN_CHARS = 400
df_test_full = apply_linguistic_filters(df_test_full, max_between_chars=MAX_BETWEEN_CHARS, pred_col='raw_hybrid_pred')
diag_counts = compute_linguistic_diagnostics(df_test_full, pred_col='raw_hybrid_pred', sanitized_col='sanitized_pred')

n_initial_pos = diag_counts['initial_positive']
n_pos_neg_only = diag_counts['rejected_negation_only']
n_pos_dist_only = diag_counts['rejected_distance_only']
n_pos_both = diag_counts['rejected_both']
n_total_rejected = diag_counts['total_unique_rejected']
n_sanitized_pos = diag_counts['surviving_positive']
assert n_sanitized_pos + n_total_rejected == n_initial_pos

prec_raw, rec_raw, f1_raw, _ = precision_recall_fscore_support(test_y_true, df_test_full['raw_hybrid_pred'], average='binary', zero_division=0)
prec_san, rec_san, f1_san, _ = precision_recall_fscore_support(test_y_true, df_test_full['sanitized_pred'], average='binary', zero_division=0)

linguistic_diagnostics_df = pd.DataFrame([{
    'initial_positive': n_initial_pos,
    'rejected_negation_only': n_pos_neg_only,
    'rejected_distance_only': n_pos_dist_only,
    'rejected_both': n_pos_both,
    'total_unique_rejected': n_total_rejected,
    'surviving_positive': n_sanitized_pos,
    'raw_pos_precision': round(float(prec_raw), 4),
    'raw_pos_recall': round(float(rec_raw), 4),
    'raw_pos_f1': round(float(f1_raw), 4),
    'sanitized_pos_precision': round(float(prec_san), 4),
    'sanitized_pos_recall': round(float(rec_san), 4),
    'sanitized_pos_f1': round(float(f1_san), 4),
}])
linguistic_diagnostics_df

,initial_positive,rejected_negation_only,rejected_distance_only,rejected_both,total_unique_rejected,surviving_positive,raw_pos_precision,raw_pos_recall,raw_pos_f1,sanitized_pos_precision,sanitized_pos_recall,sanitized_pos_f1
0,93,7,0,0,7,86,0.6129,0.57,0.5907,0.6163,0.53,0.5699


## 7. Standardized 17-Column UnifiedRelationRecord Formatting

Positive relations surviving linguistic filtering are formatted into the canonical `UnifiedRelationRecord` schema:
- Exactly 17 columns in specified order (`dl_utils.UNIFIED_RELATION_SCHEMA`).
- Stable relation ID generated via SHA-256 hash digest (`dl_utils.generate_relation_id`).
- Direction normalized strictly to `cause->effect`.

In [6]:
# Section 6: Standardized 17-Column UnifiedRelationRecord Formatting
df_pos_relations = df_test_full[df_test_full['sanitized_pred'] == 1].copy()

unified_records = []
candidate_id_to_relation_id = {}

for _, r in df_pos_relations.iterrows():
    record = dl_utils.build_unified_relation_record(
        candidate_id=str(r['candidate_id']),
        article_id=str(r['article_id']),
        section_id=str(r['section_id']),
        cause_id=str(r['arg1_id']),
        cause_text=str(r['arg1_text']),
        cause_start=int(r['arg1_start']),
        cause_end=int(r['arg1_end']),
        effect_id=str(r['arg2_id']),
        effect_text=str(r['arg2_text']),
        effect_start=int(r['arg2_start']),
        effect_end=int(r['arg2_end']),
        confidence=float(r['hybrid_prob']),
        extraction_method='hybrid_probability_fusion',
        evidence_text=str(r['between_text']),
        split='test',
        direction='cause->effect',
        provenance=f'PubMedBERT_alpha{FROZEN_ALPHA:.2f}+LogisticRegression_1to2_tau{FROZEN_TAU:.4f}',
    )
    unified_records.append(record)
    candidate_id_to_relation_id[str(r['candidate_id'])] = record['relation_id']

df_unified_records = pd.DataFrame(unified_records, columns=dl_utils.UNIFIED_RELATION_SCHEMA)
assert list(df_unified_records.columns) == dl_utils.UNIFIED_RELATION_SCHEMA, 'UnifiedRelationRecord schema mismatch'
assert len(df_unified_records) == n_sanitized_pos, 'Record count mismatch with sanitized positives'
assert (df_unified_records['direction'] == 'cause->effect').all(), 'Non-directed relation found'
assert (df_unified_records['split'] == 'test').all(), 'Split mismatch in relation records'
assert len(candidate_id_to_relation_id) == len(df_unified_records)

## 8. NetworkX DAG Construction & Deterministic Cycle Breaking

### Causal Graph Formulation
- **Nodes**: Standardized biomedical entity identifiers qualified by section: `(section_id, entity_id)`.
- **Edges**: Directed causal edges $(u \to v)$ weighted by hybrid confidence $P_{\text{hybrid}}$.
- **Acyclicity Enforcement**: Reuses Phase 1 deterministic cycle breaker (`AML/causal_extractors.py` -> `enforce_dag_acyclicity`):
  1. Identifies and prunes self-loops $(u \to u)$.
  2. Resolves bidirectional conflicts $(u \rightleftharpoons v)$ by pruning the lower-confidence edge (with deterministic tie-breaking).
  3. Breaks multi-node feedback cycles by removing minimum-confidence feedback edges.

### Edge-Level Ground Truth Evaluation
We evaluate directed edge precision, recall, and F1 before and after cycle pruning against gold relations from `data/processed/biocause_relations.csv`.

In [7]:
# Section 7: NetworkX DAG Construction & Deterministic Cycle Breaking
G_pred = nx.DiGraph()
for _, r in df_unified_records.iterrows():
    u = f"{r['section_id']}_{r['cause_id']}"
    v = f"{r['section_id']}_{r['effect_id']}"
    if not G_pred.has_node(u):
        G_pred.add_node(u, display_text=r['cause_text'], section_id=r['section_id'])
    if not G_pred.has_node(v):
        G_pred.add_node(v, display_text=r['effect_text'], section_id=r['section_id'])
    if G_pred.has_edge(u, v):
        G_pred[u][v]['evidence_count'] += 1
        G_pred[u][v]['confidence'] = max(G_pred[u][v]['confidence'], float(r['confidence']))
    else:
        G_pred.add_edge(u, v, confidence=float(r['confidence']), evidence_count=1, extraction_method=r['extraction_method'])

n_nodes_before = G_pred.number_of_nodes()
n_edges_before = G_pred.number_of_edges()
is_dag_before = nx.is_directed_acyclic_graph(G_pred)

G_dag, audit_log = enforce_dag_acyclicity(G_pred, weight_attr='confidence')
n_nodes_after = G_dag.number_of_nodes()
n_edges_after = G_dag.number_of_edges()
is_dag_after = nx.is_directed_acyclic_graph(G_dag)
assert is_dag_after, 'DAG enforcement failed to guarantee acyclicity'

n_self_loops = sum(1 for e in audit_log if e.get('reason') == 'self_loop')
n_bidirectional = sum(1 for e in audit_log if 'bidirectional_conflict' in e.get('reason', ''))
n_cycle_feedback = sum(1 for e in audit_log if 'cycle' in e.get('reason', ''))
assert n_edges_before == n_edges_after + len(audit_log)

dag_pruning_summary_df = pd.DataFrame([{
    'Nodes_Before': n_nodes_before,
    'Edges_Before': n_edges_before,
    'Is_DAG_Before': is_dag_before,
    'Self_Loops_Pruned': n_self_loops,
    'Bidirectional_Conflicts_Pruned': n_bidirectional,
    'Cycle_Feedback_Pruned': n_cycle_feedback,
    'Total_Pruned': len(audit_log),
    'Final_DAG_Nodes': n_nodes_after,
    'Final_DAG_Edges': n_edges_after,
    'Is_DAG_After': is_dag_after,
}])
display(dag_pruning_summary_df)

test_articles = set(df_aml_test.merge(df_cands_test_meta[['candidate_id', 'article_id']], on='candidate_id')['article_id'].unique())
gold_test_rels = df_gold_relations[df_gold_relations['article_id'].isin(test_articles)].copy()

G_gold = nx.DiGraph()
for _, r in gold_test_rels.iterrows():
    G_gold.add_edge(f"{r['section_id']}_{r['cause_id']}", f"{r['section_id']}_{r['effect_id']}")

def evaluate_graph_topology(G_g: nx.DiGraph, G_p: nx.DiGraph, label: str):
    gold_edges = set(G_g.edges())
    pred_edges = set(G_p.edges())
    tp = len(gold_edges & pred_edges)
    fp = len(pred_edges - gold_edges)
    fn = len(gold_edges - pred_edges)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {
        'Stage': label, 'Gold Edges': len(gold_edges), 'Pred Edges': len(pred_edges),
        'TP': tp, 'FP': fp, 'FN': fn,
        'Precision': round(p, 4), 'Recall': round(r, 4), 'F1': round(f1, 4),
    }

eval_before = evaluate_graph_topology(G_gold, G_pred, 'Before Cycle Pruning')
eval_after = evaluate_graph_topology(G_gold, G_dag, 'After DAG Enforcement')
df_graph_eval = pd.DataFrame([eval_before, eval_after])
display(df_graph_eval[['Stage', 'Gold Edges', 'Pred Edges', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1']])

,Nodes_Before,Edges_Before,Is_DAG_Before,Self_Loops_Pruned,Bidirectional_Conflicts_Pruned,Cycle_Feedback_Pruned,Total_Pruned,Final_DAG_Nodes,Final_DAG_Edges,Is_DAG_After
0,127,86,False,0,7,0,7,127,79,True


,Stage,Gold Edges,Pred Edges,TP,FP,FN,Precision,Recall,F1
0,Before Cycle Pruning,100,86,53,33,47,0.6163,0.53,0.5699
1,After DAG Enforcement,100,79,53,26,47,0.6709,0.53,0.5922


## 9. Probabilistic Bayesian Network Compilation & Interventional Queries

### 1. Structural Compilation of the Extracted DAG Topology
- The acyclic NetworkX DAG pruned from test predictions is converted into a `pgmpy.models.DiscreteBayesianNetwork(bn_edges)` instance.
- **Strict Methodological Distinction**: This compiled network captures **pure graph topology** (directed edges, parent-child structures, d-separation properties, and topological ordering). It does **NOT** possess empirically validated conditional probability distributions (CPDs), as observational text mining extracts causal assertions rather than population-level joint state frequency distributions.

### 2. Separate Illustrative Pearl $do$-Calculus Demonstration (Synthetic CPDs)
- To demonstrate formal causal reasoning ($do$-calculus vs. observational conditioning) using `pgmpy`, we define a separate, canonical 4-node regulatory pathway with synthetic, illustrative CPDs.
- This explicitly illustrates Pearl's graph mutilation ($do$-operator) versus Bayesian conditioning without falsely claiming that the large-scale mined graph has parameter-fitted CPDs.

In [8]:
# Section 8: Bayesian Network Compilation & Interventional Queries
# Part 1: Structural Compilation of Extracted DAG Topology (pure topology; no empirical CPDs claimed)
bn_model = compile_discrete_bayesian_network(G_dag)
n_isolated = len(list(nx.isolates(bn_model)))
assert len(bn_model.nodes()) == n_nodes_after and len(bn_model.edges()) == n_edges_after

# Part 2: Separate Illustrative Pearl do-Calculus Demonstration (Synthetic CPDs)
demo_bn = DiscreteBayesianNetwork([
    ('Sensor_Kinase', 'Response_Regulator'),
    ('Co_Activator', 'Response_Regulator'),
    ('Response_Regulator', 'Antibiotic_Resistance'),
])
cpd_sensor = TabularCPD(variable='Sensor_Kinase', variable_card=2, values=[[0.70], [0.30]], state_names={'Sensor_Kinase': ['0', '1']})
cpd_coact = TabularCPD(variable='Co_Activator', variable_card=2, values=[[0.75], [0.25]], state_names={'Co_Activator': ['0', '1']})
cpd_reg = TabularCPD(
    variable='Response_Regulator', variable_card=2,
    values=[[0.90, 0.40, 0.30, 0.05], [0.10, 0.60, 0.70, 0.95]],
    evidence=['Sensor_Kinase', 'Co_Activator'], evidence_card=[2, 2],
    state_names={'Response_Regulator': ['0', '1'], 'Sensor_Kinase': ['0', '1'], 'Co_Activator': ['0', '1']},
)
cpd_res = TabularCPD(
    variable='Antibiotic_Resistance', variable_card=2,
    values=[[0.95, 0.15], [0.05, 0.85]],
    evidence=['Response_Regulator'], evidence_card=[2],
    state_names={'Antibiotic_Resistance': ['0', '1'], 'Response_Regulator': ['0', '1']},
)
demo_bn.add_cpds(cpd_sensor, cpd_coact, cpd_reg, cpd_res)
assert demo_bn.check_model(), 'Illustrative Bayesian network check_model failed'

infer_obs = VariableElimination(demo_bn)
p_base = float(infer_obs.query(['Antibiotic_Resistance']).values[1])
p_obs = float(infer_obs.query(['Antibiotic_Resistance'], evidence={'Sensor_Kinase': '1'}).values[1])
interv_model = demo_bn.do(['Response_Regulator'])
p_interv = float(VariableElimination(interv_model).query(['Antibiotic_Resistance'], evidence={'Response_Regulator': '1'}).values[1])

bn_inference_df = pd.DataFrame([
    {'Query (Illustrative Synthetic CPDs)': 'Baseline P(Resistance = 1)', 'Probability': round(p_base, 4), 'Delta_vs_Baseline': 0.0},
    {'Query (Illustrative Synthetic CPDs)': 'Observational P(Resistance = 1 | Sensor = 1)', 'Probability': round(p_obs, 4), 'Delta_vs_Baseline': round(p_obs - p_base, 4)},
    {'Query (Illustrative Synthetic CPDs)': 'Interventional P(Resistance = 1 | do(Regulator = 1))', 'Probability': round(p_interv, 4), 'Delta_vs_Baseline': round(p_interv - p_base, 4)},
])
bn_inference_df

,Query (Illustrative Synthetic CPDs),Probability,Delta_vs_Baseline
0,Baseline P(Resistance = 1),0.359,0.000
1,Observational P(Resistance = 1 | Sensor = 1),0.660,0.301
2,Interventional P(Resistance = 1 | do(Regulator...,0.850,0.491


## 10. Downstream Artifact Export for Task 5

We export standardized artifacts to `data/processed/` for downstream Task 5 evaluation and ablation:
1. `dl_biocause_hybrid_validation_predictions.csv`: 1,884 candidates with AML, DL, and Hybrid probabilities.
2. `dl_biocause_hybrid_test_predictions.csv`: 1,394 candidates with AML, DL, and Hybrid probabilities.
3. `dl_biocause_hybrid_positive_relations.csv`: 17-column `UnifiedRelationRecord` format.
4. `dl_biocause_hybrid_dag_edges.csv`: Final acyclic DAG edge list.
5. `dl_biocause_fusion_config.json`: Selected $\alpha^*$, threshold $\tau^*$, and optimization metadata.
6. `dl_biocause_graph_pruning_diagnostics.json`: Audit log of cycle-pruning and linguistic filtering.

In [9]:
# Section 9: Downstream Artifact Export & Verification
# 1. Export hybrid validation predictions (1,884 rows)
val_hybrid_export = pd.DataFrame({
    'candidate_id': df_aml_val['candidate_id'].values,
    'true_label': val_y_true,
    'aml_prob': val_p_aml,
    'dl_prob': val_p_dl,
    'hybrid_prob': val_p_hybrid_best,
    'aml_pred': df_aml_val['aml_pred'].values,
    'dl_pred': df_dl_val['dl_pred'].values,
    'hybrid_pred': (val_p_hybrid_best >= FROZEN_TAU).astype(int),
    'split': 'validation',
})
val_exp_path = DATA_DIR / 'dl_biocause_hybrid_validation_predictions.csv'
dl_utils.export_predictions(val_hybrid_export, str(val_exp_path), format_type='csv')
verify_prediction_export(
    filepath=val_exp_path,
    expected_rows=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['validation']['total'],
    split_name='validation',
    source_candidate_ids=df_aml_val['candidate_id'].values,
    source_true_labels=val_y_true,
    source_aml_probs=val_p_aml,
    source_dl_probs=val_p_dl,
    source_aml_preds=df_aml_val['aml_pred'].values,
    source_dl_preds=df_dl_val['dl_pred'].values,
    frozen_alpha=FROZEN_ALPHA,
    frozen_tau=FROZEN_TAU,
)

# 2. Export hybrid test predictions (1,394 rows)
test_hybrid_export = pd.DataFrame({
    'candidate_id': df_aml_test['candidate_id'].values,
    'true_label': test_y_true,
    'aml_prob': test_p_aml,
    'dl_prob': test_p_dl,
    'hybrid_prob': test_p_hybrid,
    'aml_pred': df_aml_test['aml_pred'].values,
    'dl_pred': df_dl_test['dl_pred'].values,
    'hybrid_pred': test_y_pred_hybrid,
    'sanitized_pred': df_test_full['sanitized_pred'].values,
    'split': 'test',
})
test_exp_path = DATA_DIR / 'dl_biocause_hybrid_test_predictions.csv'
dl_utils.export_predictions(test_hybrid_export, str(test_exp_path), format_type='csv')
verify_prediction_export(
    filepath=test_exp_path,
    expected_rows=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total'],
    split_name='test',
    source_candidate_ids=df_aml_test['candidate_id'].values,
    source_true_labels=test_y_true,
    source_aml_probs=test_p_aml,
    source_dl_probs=test_p_dl,
    source_aml_preds=df_aml_test['aml_pred'].values,
    source_dl_preds=df_dl_test['dl_pred'].values,
    frozen_alpha=FROZEN_ALPHA,
    frozen_tau=FROZEN_TAU,
    include_sanitized_col=True,
)

# 3. Export unified relation records (17 columns)
rels_exp_path = DATA_DIR / 'dl_biocause_hybrid_positive_relations.csv'
dl_utils.export_predictions(df_unified_records, str(rels_exp_path), format_type='csv')
verify_relation_export(
    filepath=rels_exp_path,
    expected_count=n_sanitized_pos,
    sanitized_candidate_ids=df_pos_relations['candidate_id'].tolist(),
    traceability_map=candidate_id_to_relation_id,
)

# 4. Export final DAG edge list
dag_edges_records = [
    {
        'source': u,
        'target': v,
        'confidence': d.get('confidence', 1.0),
        'evidence_count': d.get('evidence_count', 1),
        'extraction_method': d.get('extraction_method', 'hybrid_probability_fusion'),
    }
    for u, v, d in G_dag.edges(data=True)
]
df_dag_edges = pd.DataFrame(
    dag_edges_records,
    columns=['source', 'target', 'confidence', 'evidence_count', 'extraction_method'],
)
dag_exp_path = DATA_DIR / 'dl_biocause_hybrid_dag_edges.csv'
dl_utils.export_predictions(df_dag_edges, str(dag_exp_path), format_type='csv')

# 5. Export fusion configuration JSON
assert eval_hybrid_selected['Stream'].startswith('Hybrid Model'), (
    f"Exported model metrics identity is not the selected hybrid model: {eval_hybrid_selected['Stream']}"
)
def serialize_test_metrics(eval_dict: Dict[str, Any]) -> Dict[str, Any]:
    return {
        k: float(v) if isinstance(v, (int, float, np.floating)) else v
        for k, v in eval_dict.items()
        if k != 'Confusion Matrix'
    }

hybrid_test_metrics_export = serialize_test_metrics(eval_hybrid_selected)
val_delta_pos_f1_vs_aml = float(best_val_f1_pos - aml_val_pos_f1)
val_delta_macro_f1_vs_aml = float(best_val_macro_f1 - aml_val_macro_f1)
test_delta_pos_f1_vs_aml = float(eval_hybrid_selected['Pos F1'] - eval_aml_validation['Pos F1'])
test_delta_macro_f1_vs_aml = float(eval_hybrid_selected['Macro F1'] - eval_aml_validation['Macro F1'])

fusion_config = {
    'task': 'DL_04_hybrid_graph_integration',
    'selected_model': eval_hybrid_selected['Stream'],
    'alpha_star': float(FROZEN_ALPHA),
    'threshold_star': float(FROZEN_TAU),
    'aml_threshold_star': float(AML_VALIDATION_TAU),
    'dl_threshold_star': float(DL_VALIDATION_TAU),
    'optimization_objective': (
        'maximize_validation_positive_f1_then_macro_f1_then_smaller_alpha_then_closest_to_0.50_then_higher_tau'
    ),
    'validation_metrics': {
        'positive_f1': float(best_val_f1_pos),
        'macro_f1': float(best_val_macro_f1),
    },
    'aml_only_validation_metrics': {
        'threshold': float(AML_VALIDATION_TAU),
        'positive_f1': float(aml_val_pos_f1),
        'macro_f1': float(aml_val_macro_f1),
    },
    'dl_only_validation_metrics': {
        'threshold': float(DL_VALIDATION_TAU),
        'positive_f1': float(dl_val_pos_f1),
        'macro_f1': float(dl_val_macro_f1),
    },
    'delta_positive_f1_vs_aml': test_delta_pos_f1_vs_aml,
    'delta_macro_f1_vs_aml': test_delta_macro_f1_vs_aml,
    'dl_contribution_summary': {
        'uses_dl_signal': bool(FROZEN_ALPHA > 0.0),
        'validation_delta_pos_f1_vs_aml': val_delta_pos_f1_vs_aml,
        'validation_delta_macro_f1_vs_aml': val_delta_macro_f1_vs_aml,
        'test_delta_pos_f1_vs_aml': test_delta_pos_f1_vs_aml,
        'test_delta_macro_f1_vs_aml': test_delta_macro_f1_vs_aml,
        'improves_val_pos_f1_over_aml': bool(val_delta_pos_f1_vs_aml > 1e-8),
        'improves_test_pos_f1_over_aml': bool(test_delta_pos_f1_vs_aml > 1e-8),
    },
    'baseline_test_metrics': {
        'aml_fixed_0_50': serialize_test_metrics(eval_aml),
        'aml_validation_selected': serialize_test_metrics(eval_aml_validation),
        'dl_validation_selected': serialize_test_metrics(eval_dl_validation),
        'dl_fixed_0_50': serialize_test_metrics(eval_dl_default),
    },
    'test_metrics': hybrid_test_metrics_export,
}
config_exp_path = DATA_DIR / 'dl_biocause_fusion_config.json'
with open(config_exp_path, 'w') as f:
    json.dump(fusion_config, f, indent=2)
assert config_exp_path.exists()

# 6. Export graph pruning diagnostics JSON
diagnostics_export = {
    'initial_candidate_edges': n_edges_before,
    'final_dag_edges': n_edges_after,
    'pruned_edges_count': len(audit_log),
    'pruning_breakdown': {
        'self_loops': n_self_loops,
        'bidirectional_conflicts': n_bidirectional,
        'directed_cycle_feedback': n_cycle_feedback,
    },
    'linguistic_filtering': diag_counts,
    'audit_log': audit_log,
    'edge_evaluation': {
        'before_pruning': eval_before,
        'after_pruning': eval_after,
    },
}
diag_exp_path = DATA_DIR / 'dl_biocause_graph_pruning_diagnostics.json'
with open(diag_exp_path, 'w') as f:
    json.dump(diagnostics_export, f, indent=2, default=lambda x: str(x) if hasattr(x, '__str__') else x)
assert diag_exp_path.exists()
verify_dag_and_pruning_diagnostics(G_pred, G_dag, audit_log, diag_exp_path, dag_edges_filepath=dag_exp_path)

assert len(df_aml_val) == len(df_dl_val) == 1884
assert len(df_aml_test) == len(df_dl_test) == 1394
assert (df_aml_val['candidate_id'].to_numpy() == df_dl_val['candidate_id'].to_numpy()).all()
assert (df_aml_test['candidate_id'].to_numpy() == df_dl_test['candidate_id'].to_numpy()).all()
assert any(np.isclose(AML_VALIDATION_TAU, t) for t in build_candidate_thresholds(val_p_aml))
assert any(np.isclose(DL_VALIDATION_TAU, t) for t in build_candidate_thresholds(val_p_dl))
assert any(np.isclose(FROZEN_TAU, t) for t in build_candidate_thresholds(val_p_hybrid_best))

## 11. AML Classifier Serialization (`aml_classifier.pkl`) & Deployable Hybrid Model Packaging

Packages the complete hybrid BioCause inference bundle into `DL/models/hybrid_biocause/`:
- `aml_classifier.pkl`: Joblib-serialized sklearn `Pipeline` (`BioCauseAMLFeatureExtractor` + `LogisticRegression(class_weight={0: 1.0, 1: 2.0}, C=1.0, max_iter=200, random_state=42)`) fitted strictly on training data and verified against exported validation/test `aml_prob` streams.
- `pubmedbert/`: Frozen PubMedBERT SafeTensors weights and special-marker tokenizer (copied or converted from `DL_03` outputs without retraining).
- `fusion_config.json` & `model_manifest.json`: Dynamic frozen configuration derived directly from `data/processed/dl_biocause_fusion_config.json` and `dl_biocause_metrics.json`.

In [ ]:
# Section 11: AML Pipeline Serialization (aml_classifier.pkl) & Hybrid Model Packaging
import importlib.metadata
import json
import os
import shutil
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

if 'ROOT_DIR' not in globals():
    _curr = Path(os.getcwd()).resolve()
    ROOT_DIR = next(
        (
            cand.resolve()
            for base in [Path('/content/causal-nlp-reasoning'), Path('/workspace/causal-nlp-reasoning'), Path('/kaggle/working/causal-nlp-reasoning'), _curr]
            if base.exists()
            for cand in [base, *base.parents]
            if (cand / 'AML' / 'causal_extractors.py').is_file() and (cand / 'DL' / 'dl_utils.py').is_file()
        ),
        None,
    )
    if ROOT_DIR is None:
        raise FileNotFoundError('Repository root not found.')
DATA_DIR = ROOT_DIR / 'data' / 'processed'
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
from hybrid_inference import (
    REQUIRED_AML_INPUT_COLUMNS,
    collect_artifact_inventory,
    fit_and_serialize_aml_pipeline,
    persist_and_verify_pubmedbert,
)

dl_utils.set_seed(42)
HYBRID_PACKAGE_DIR = ROOT_DIR / 'DL' / 'models' / 'hybrid_biocause'
HYBRID_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
AML_PKL_PATH = HYBRID_PACKAGE_DIR / 'aml_classifier.pkl'
HYBRID_PUBMED_DIR = HYBRID_PACKAGE_DIR / 'pubmedbert'

# 1. Fit Logistic Regression (Moderate Weight 1:2) pipeline strictly on train split and dump via Joblib
candidates_csv = DATA_DIR / 'biocause_candidates.csv'
aml_pipeline = fit_and_serialize_aml_pipeline(candidates_csv, AML_PKL_PATH)

# 2. Verify exact alignment against frozen validation and test AML probabilities (read-only verification)
df_cands_all = pd.read_csv(candidates_csv, keep_default_na=False).set_index('candidate_id')
df_aml_val_ref = pd.read_csv(DATA_DIR / 'aml_biocause_validation_predictions.csv')
df_aml_test_ref = pd.read_csv(DATA_DIR / 'aml_biocause_test_predictions.csv')

val_aml_probs_pkg = aml_pipeline.predict_proba(df_cands_all.loc[df_aml_val_ref['candidate_id']].reset_index())[:, 1]
test_aml_probs_pkg = aml_pipeline.predict_proba(df_cands_all.loc[df_aml_test_ref['candidate_id']].reset_index())[:, 1]
assert np.allclose(val_aml_probs_pkg, df_aml_val_ref['aml_prob'].to_numpy(), atol=1e-6), 'Packaged AML val prob mismatch'
assert np.allclose(test_aml_probs_pkg, df_aml_test_ref['aml_prob'].to_numpy(), atol=1e-6), 'Packaged AML test prob mismatch'

# 3. Read actual frozen fusion & DL_03 metrics configs (never hardcode)
fusion_src = json.loads((DATA_DIR / 'dl_biocause_fusion_config.json').read_text())
dl_metrics_path = DATA_DIR / 'dl_biocause_metrics.json'
dl_metrics_src = json.loads(dl_metrics_path.read_text()) if dl_metrics_path.is_file() else {}

# 4. Ensure frozen PubMedBERT SafeTensors artifacts exist inside hybrid_biocause/pubmedbert/
best_dl_dir = ROOT_DIR / 'DL' / 'models' / 'biocause_pubmedbert_best'
selected_ckpt_name = str(dl_metrics_src.get('selected_checkpoint_name', 'checkpoint-3723'))
ckpt_fallback_dir = ROOT_DIR / 'DL' / 'checkpoints' / 'biocause_pubmedbert' / selected_ckpt_name
val_manifest_df = pd.read_csv(DATA_DIR / 'dl_biocause_val.csv')
sanity_texts = val_manifest_df['marked_text'].iloc[:4].astype(str).tolist()

def _has_hf_weights(d: Path) -> bool:
    return (d / 'config.json').is_file() and ((d / 'model.safetensors').is_file() or (d / 'pytorch_model.bin').is_file())

if _has_hf_weights(HYBRID_PUBMED_DIR):
    pubmed_saved = True
elif _has_hf_weights(best_dl_dir):
    shutil.copytree(best_dl_dir, HYBRID_PUBMED_DIR, dirs_exist_ok=True)
    pubmed_saved = _has_hf_weights(HYBRID_PUBMED_DIR)
elif _has_hf_weights(ckpt_fallback_dir):
    persist_and_verify_pubmedbert(
        source_checkpoint_dir=ckpt_fallback_dir,
        output_dirs=[best_dl_dir, HYBRID_PUBMED_DIR],
        sample_texts=sanity_texts,
        max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
    )
    pubmed_saved = _has_hf_weights(HYBRID_PUBMED_DIR)
else:
    pubmed_saved = False
    print(f'Notice: Frozen PubMedBERT weights not yet saved on disk ({HYBRID_PUBMED_DIR}). Run DL_03 Section 9 or Section 10 recovery cell.')

# 5. Write package fusion_config.json and model_manifest.json from actual outputs
pkg_fusion_config = {
    'task': 'DL_04_hybrid_graph_integration',
    'package_name': 'hybrid_biocause',
    'selected_model': fusion_src['selected_model'],
    'alpha_star': float(fusion_src['alpha_star']),
    'threshold_star': float(fusion_src['threshold_star']),
    'aml_threshold_star': float(fusion_src['aml_threshold_star']),
    'aml_default_threshold': 0.5,
    'dl_threshold_star': float(fusion_src['dl_threshold_star']),
    'fusion_formula': 'P_hybrid = alpha_star * P_DL + (1.0 - alpha_star) * P_AML',
    'selection_split': 'validation',
    'selection_split_rows': len(df_aml_val_ref),
    'optimization_objective': fusion_src.get('optimization_objective', ''),
    'models': {
        'dl_stream': {
            'model_name': dl_utils.PUBMEDBERT_CHECKPOINT,
            'artifact_subdir': 'pubmedbert',
            'selected_checkpoint': selected_ckpt_name,
            'selected_epoch': float(dl_metrics_src.get('selected_epoch', 3.0)),
            'max_length': dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
            'special_entity_markers': list(dl_utils.ENTITY_MARKERS),
        },
        'aml_stream': {
            'model_name': 'Logistic Regression (Moderate Weight 1:2)',
            'artifact_file': 'aml_classifier.pkl',
            'serialization': 'joblib',
            'estimator_class': 'sklearn.pipeline.Pipeline',
            'required_input_columns': list(REQUIRED_AML_INPUT_COLUMNS),
        },
    },
    'post_processing': {
        'linguistic_filter': {'enabled': True, 'max_between_chars': 400},
        'dag_cycle_breaking': {'enabled': True, 'function': 'AML/causal_extractors.py::enforce_dag_acyclicity'},
    },
    'validation_metrics': fusion_src.get('validation_metrics', {}),
    'test_metrics_reference': fusion_src.get('test_metrics', {}),
}
(HYBRID_PACKAGE_DIR / 'fusion_config.json').write_text(json.dumps(pkg_fusion_config, indent=2) + '\n')

pkg_versions = {}
for dist_name in ('torch', 'transformers', 'scikit-learn', 'joblib', 'scipy', 'pandas', 'numpy', 'networkx', 'safetensors'):
    try:
        pkg_versions[dist_name] = importlib.metadata.version(dist_name)
    except importlib.metadata.PackageNotFoundError:
        pkg_versions[dist_name] = 'unknown'

df_train_raw = df_cands_all[df_cands_all['split'] == 'train']
manifest_data = {
    'package_name': 'hybrid_biocause',
    'package_directory': 'DL/models/hybrid_biocause',
    'persistence_status': {
        'pubmedbert_weights_saved': bool(pubmed_saved),
        'aml_classifier_pkl_saved': bool(AML_PKL_PATH.is_file()),
        'reload_and_inference_verified': False,
    },
    'dataset_splits': {
        'biocause_model_eligible': dl_metrics_src.get('split_counts', {
            'train': {'total': 9928, 'positive': 621, 'negative': 9307},
            'validation': {'total': 1884, 'positive': 131, 'negative': 1753},
            'test': {'total': 1394, 'positive': 100, 'negative': 1294},
        }),
        'biocause_raw_train_fit_rows': int(len(df_train_raw)),
        'dynamic_pos_weight': 9307 / 621,
    },
    'selected_checkpoint': selected_ckpt_name,
    'selected_epoch': float(dl_metrics_src.get('selected_epoch', 3.0)),
    'fusion_configuration': {
        'alpha_star': float(fusion_src['alpha_star']),
        'threshold_star': float(fusion_src['threshold_star']),
        'dl_threshold_star': float(fusion_src['dl_threshold_star']),
        'aml_threshold_star': float(fusion_src['aml_threshold_star']),
    },
    'artifacts': collect_artifact_inventory(HYBRID_PACKAGE_DIR),
    'package_versions': pkg_versions,
}
(HYBRID_PACKAGE_DIR / 'model_manifest.json').write_text(json.dumps(manifest_data, indent=2) + '\n')
pd.DataFrame(collect_artifact_inventory(HYBRID_PACKAGE_DIR))

## 12. Hybrid Model Package Reload & Deterministic Inference Verification (Validation Slice, No Test Labels)

Reloads `aml_classifier.pkl`, `pubmedbert/`, `fusion_config.json`, and `model_manifest.json` from `DL/models/hybrid_biocause/`, verifies exact prediction agreement on an unlabeled 8-row validation slice against exported hybrid validation predictions (zero test labels used), updates `model_manifest.json` verification status, and displays packaged file sizes.

In [ ]:
# Section 12: Package Reload & Validation Inference Sanity Check (No Test Labels)
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

if 'ROOT_DIR' not in globals():
    _curr = Path(os.getcwd()).resolve()
    ROOT_DIR = next(
        (
            cand.resolve()
            for base in [Path('/content/causal-nlp-reasoning'), Path('/workspace/causal-nlp-reasoning'), Path('/kaggle/working/causal-nlp-reasoning'), _curr]
            if base.exists()
            for cand in [base, *base.parents]
            if (cand / 'AML' / 'causal_extractors.py').is_file() and (cand / 'DL' / 'dl_utils.py').is_file()
        ),
        None,
    )
    if ROOT_DIR is None:
        raise FileNotFoundError('Repository root not found.')
DATA_DIR = ROOT_DIR / 'data' / 'processed'
HYBRID_PACKAGE_DIR = ROOT_DIR / 'DL' / 'models' / 'hybrid_biocause'
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from hybrid_inference import HybridBioCausePredictor, collect_artifact_inventory

# 1. Reload Joblib AML pipeline and inspect prediction interface
aml_pkl_path = HYBRID_PACKAGE_DIR / 'aml_classifier.pkl'
reloaded_aml = joblib.load(aml_pkl_path)
assert hasattr(reloaded_aml, 'predict_proba') and hasattr(reloaded_aml, 'predict')

# 2. Prepare unlabeled 8-row validation slice (strictly no test split or labels)
val_ref = pd.read_csv(DATA_DIR / 'dl_biocause_hybrid_validation_predictions.csv').iloc[:8].copy()
val_manifest = pd.read_csv(DATA_DIR / 'dl_biocause_val.csv', usecols=['candidate_id', 'marked_text'])
cands_meta = pd.read_csv(DATA_DIR / 'biocause_candidates.csv', keep_default_na=False)
val_slice = (
    val_ref[['candidate_id']]
    .merge(val_manifest, on='candidate_id', how='inner')
    .merge(
        cands_meta[['candidate_id', 'char_distance', 'arg1_before_arg2', 'between_text', 'arg1_text', 'arg2_text']],
        on='candidate_id',
        how='inner',
    )
)
assert len(val_slice) == 8 and val_slice['candidate_id'].tolist() == val_ref['candidate_id'].tolist()

# 3. Verify reloaded AML pipeline alone matches reference validation aml_prob
aml_slice_probs = reloaded_aml.predict_proba(val_slice)[:, 1]
assert np.allclose(aml_slice_probs, val_ref['aml_prob'].to_numpy(), atol=1e-6)

# 4. Verify full HybridBioCausePredictor (requires pubmedbert/ weights saved via DL_03 Section 9 or 10)
if not (HYBRID_PACKAGE_DIR / 'pubmedbert' / 'config.json').is_file():
    raise FileNotFoundError(
        f"Missing persisted PubMedBERT artifacts in {HYBRID_PACKAGE_DIR / 'pubmedbert'}. "
        'Run DL_03 Section 9 or Section 10 recovery cell in Colab first.'
    )

predictor = HybridBioCausePredictor(HYBRID_PACKAGE_DIR)
slice_preds = predictor.predict_candidates(val_slice, batch_size=8)
assert slice_preds['candidate_id'].tolist() == val_ref['candidate_id'].tolist()
assert np.allclose(slice_preds['aml_prob'].to_numpy(), val_ref['aml_prob'].to_numpy(), atol=1e-6)
assert np.allclose(slice_preds['dl_prob'].to_numpy(), val_ref['dl_prob'].to_numpy(), atol=1e-4)
assert np.allclose(slice_preds['hybrid_prob'].to_numpy(), val_ref['hybrid_prob'].to_numpy(), atol=1e-4)
assert np.array_equal(slice_preds['hybrid_pred'].to_numpy(), val_ref['hybrid_pred'].to_numpy())

# 5. Record verified reload status in model_manifest.json
manifest_path = HYBRID_PACKAGE_DIR / 'model_manifest.json'
manifest_obj = json.loads(manifest_path.read_text())
manifest_obj['persistence_status'] = {
    'pubmedbert_weights_saved': True,
    'aml_classifier_pkl_saved': True,
    'reload_and_inference_verified': True,
}
manifest_obj['artifacts'] = collect_artifact_inventory(HYBRID_PACKAGE_DIR)
manifest_path.write_text(json.dumps(manifest_obj, indent=2) + '\n')

display(slice_preds[['candidate_id', 'aml_prob', 'dl_prob', 'hybrid_prob', 'hybrid_pred', 'sanitized_pred']].round(4))
pd.DataFrame(manifest_obj['artifacts'])